# Stop Times Checks

Checks trip duplication, stop_sequence regularity, and route-following behavior in stop_times.

In [ ]:
import csv
import importlib
import os
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import sys
from typing import Dict, List, Set, Tuple

_current = Path.cwd().resolve()
for _candidate in [_current, *_current.parents]:
    if (_candidate / "data_validation" / "gtfs_utils.py").exists():
        _project_root = _candidate
        break
else:
    raise FileNotFoundError("data_validation/gtfs_utils.py not found.")

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import scripts.basics as basics
import data_validation.gtfs_utils as gtfs_utils
gtfs_utils = importlib.reload(gtfs_utils)
from data_validation.gtfs_utils import (
    DUPLICATED_TRIPS_BASE,
    STOP_SEQUENCE_BASE,
    STOP_TIMES_SUBWAY_FILE,
    STOP_TIMES_CLEANED_FILE,
    STOP_TIMES_FILE,
    STOPS_FILE,
    TRIPS_SUBWAY_FILE,
    TRIPS_FILE,
    TRIP_IDS_TO_ELIMINATE_FILE,
    WRONG_STOP_SEQUENCES_FILE,
    check_missing_files,
    print_file_disclaimer,
    load_stop_names,
    load_trip_ids,
    load_nonempty_lines,
    sniff_dialect,
    read_dict_rows,
    build_expected_adjacency,
    collect_trip_stop_ids,
    make_signature,
    check_trip,
)

#### Duplicate full trip_id blocs in stop_times

In [ ]:
def main():
    """Find and export groups of trip_id with identical stop_times content."""
    trip_id_set = None
    trips_rows: Dict[str, List[Tuple[int, str, str, str]]] = {}
    total_rows = 0
    signature_to_trip_ids: Dict[Tuple[Tuple[int, str, str, str], ...], List[str]] = {}
    max_workers = 0
    duplicate_groups = None
    total_duplicate_trip_ids = 0
    duplicate_group_sizes = None
    trip_ids_to_eliminate: List[str] = []
    eliminate_path = None
    check_missing_files([STOP_TIMES_SUBWAY_FILE, TRIPS_SUBWAY_FILE])

    print_file_disclaimer([
        (STOP_TIMES_SUBWAY_FILE, 'stop_times'),
        (TRIPS_SUBWAY_FILE, 'trips'),
    ])

    trip_id_set = load_trip_ids(TRIPS_SUBWAY_FILE)

    print(f"Unique trip_id in 'trips': {len(trip_id_set)}")

    trips_rows = defaultdict(list)
    for row in read_dict_rows(STOP_TIMES_SUBWAY_FILE):
        total_rows += 1
        trip_id = row.get("trip_id", "")
        if trip_id not in trip_id_set:
            continue
        sequence_text = row.get("stop_sequence", "")
        arrival_time = row.get("arrival_time", "")
        departure_time = row.get("departure_time", "")
        stop_id = row.get("stop_id", "")
        try:
            stop_sequence = int(sequence_text)
        except Exception:
            stop_sequence = 10**9
        trips_rows[trip_id].append((stop_sequence, arrival_time, departure_time, stop_id))

    print(f"Total rows read from 'stop_times': {total_rows}")
    print(f"trip_id with at least one row in 'stop_times': {len(trips_rows)}")

    signature_to_trip_ids = defaultdict(list)
    max_workers = min(8, (os.cpu_count() or 4))
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(make_signature, item): item[0] for item in trips_rows.items()
        }
        for future in as_completed(futures):
            trip_id, signature = future.result()
            signature_to_trip_ids[signature].append(trip_id)

    duplicate_groups = [
        (signature, sorted(trip_ids_for_signature))
        for signature, trip_ids_for_signature in signature_to_trip_ids.items()
        if len(trip_ids_for_signature) >= 2
    ]
    duplicate_groups.sort(key=lambda item: (len(item[1]), item[1]))

    if not duplicate_groups:
        print("No pair/group of trip_id with identical sequence and schedules was found.")
        return

    total_duplicate_trip_ids = sum(
        len(trip_ids_for_signature) for _, trip_ids_for_signature in duplicate_groups
    )
    print(
        f"\nFound {len(duplicate_groups)} groups of trip_id with identical content "
        f"({total_duplicate_trip_ids} trip_id in total)."
    )

    duplicate_group_sizes = {}
    for _, trip_ids_for_signature in duplicate_groups:
        group_size = len(trip_ids_for_signature)
        duplicate_group_sizes[group_size] = duplicate_group_sizes.get(group_size, 0) + 1
    for group_size in sorted(duplicate_group_sizes):
        print(f"Groups with {group_size} trip_id: {duplicate_group_sizes[group_size]}")
        for _, trip_ids_for_signature in duplicate_groups:
            if len(trip_ids_for_signature) == group_size:
                suffix = "..." if len(trip_ids_for_signature) > 5 else ""
                print(
                    f"  Example group with {group_size} trip_id:"
                    f" {trip_ids_for_signature[:5]}{suffix}"
                )
                break

    for _, trip_ids_for_signature in duplicate_groups:
        trip_ids_to_eliminate.extend(trip_ids_for_signature[1:])

    eliminate_path = os.path.join(DUPLICATED_TRIPS_BASE, "trip_ids_to_eliminate.txt")
    with open(eliminate_path, "w", encoding="utf-8") as file_handle:
        for trip_id in sorted(trip_ids_to_eliminate):
            file_handle.write(trip_id + "\n")

    print(
        f"\nTrip_id kept from groups (1 per group):"
        f" {total_duplicate_trip_ids - len(trip_ids_to_eliminate)}"
    )
    print(f"Trip_id to eliminate: {len(trip_ids_to_eliminate)}")
    print(
        f"Total valid trip_id after removing duplicates:"
        f" {len(trip_id_set) - len(trip_ids_to_eliminate)}"
    )
    print(
        f"{Path(TRIP_IDS_TO_ELIMINATE_FILE).name} generated in"
        f" {Path(TRIP_IDS_TO_ELIMINATE_FILE).relative_to(_project_root).parent}"
    )

main()

With `.src/gtfs/data/2_duplicated_trips/trip_ids_to_eliminate.txt` created in this cell, we create `trips_cleaned.txt` and `stop_times_cleaned.txt` in the same folder with `data_validation/processing/2_duplicated_trips.py`

#### Does stop_sequence increment by one?

Goal: check whether stop_sequence in stop_times.txt increments by one. To do this, we read stop_times once and aggregate the stop_sequence by trip_id for each trip_id in trips.txt. 

In [ ]:
def main():
    """Check that each trip has stop_sequence values that increment by one."""
    trip_ids = None
    seq_by_trip: Dict[str, Set[int]] = {}
    dict_seq: Dict[str, List[int]] = {}
    max_workers = 0
    violations_total = 0
    check_missing_files([STOP_TIMES_CLEANED_FILE, TRIPS_FILE])

    print_file_disclaimer([
        (STOP_TIMES_CLEANED_FILE, 'stop_times'),
        (TRIPS_FILE, 'trips'),
    ])

    trip_ids = load_trip_ids(TRIPS_FILE)

    print(f"Number of trip_id in 'trips': {len(trip_ids)}")

    # Build: trip_id -> set of stop_sequence (no duplicates), in a single pass over stop_times_cleaned
    seq_by_trip = defaultdict(set)
    for r in read_dict_rows(STOP_TIMES_CLEANED_FILE):
        tid = r.get('trip_id', '')
        if tid not in trip_ids:
            continue
        seq_str = r.get('stop_sequence', '')
        try:
            seq = int(seq_str)
        except Exception:
            # Ignore non-numeric or empty values
            continue
        seq_by_trip[tid].add(seq)

    # Final map: trip_id -> sorted list of stop_sequence (no duplicates)
    dict_seq = {}
    for trip_id in trip_ids:
        dict_seq[trip_id] = sorted(seq_by_trip.get(trip_id, set()))

    # Parallel validation
    max_workers = min(8, (os.cpu_count() or 4))
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(check_trip, trip_id, dict_seq[trip_id]): trip_id for trip_id in trip_ids
        }
        for fut in as_completed(futures):
            msgs = fut.result()
            violations_total += len(msgs)
            for m in msgs:
                print(m)

    if violations_total == 0:
        print(
            "Correct: all trips in 'trips' have stop_sequence in 'stop_times'"
            " that increments by one."
        )
    else:
        print(f"Total violations detected: {violations_total}")

main()

#### The canonical stop sequence is followed correctly?

##### Verifying it for the original data

Mechanism: We use dictionaries from scripts/basics.py. Each route_name has a route_id in 'subway_routes_names_ids'. For each route_id, we gather trip_id values (and direction_id) from 'trips_subway_cleaned.txt'. For each trip_id, we read rows from 'stop_times_subway_cleaned.txt' ordered by stop_sequence and compare each consecutive pair with the expected order from 'subway_route_names_stop_ids' (reversed when direction_id=1). If a pair is flagged, we print only the two stops in bad order with their stop_sequence values and stop names from 'stops_subway_cleaned.txt'.

In [ ]:
def build_trip_to_route_dir(trips_file, rid_to_name):
    """Read trips and return a mapping of trip_id to (route_id, direction_id).

    args:
        trips_file: Path to the trips CSV file.
        rid_to_name: Mapping from route_id to route short name.

    returns:
        Tuple of the mapping dict and the count of matched rows.
    """
    trip_to_route_dir = {}
    matched_rows = 0
    for row in read_dict_rows(trips_file):
        trip_id = row.get("trip_id", "").strip()
        if not trip_id:
            continue
        route_id = row.get("route_id", "").strip()
        if route_id not in rid_to_name:
            continue
        matched_rows += 1
        trip_to_route_dir[trip_id] = (route_id, row.get("direction_id", "").strip() or "")
    return trip_to_route_dir, matched_rows


def build_trip_seq_rows(stop_times_file, trip_ids):
    """Read stop_times and return a mapping of trip_id to sorted (seq, stop_id) rows.

    args:
        stop_times_file: Path to the stop_times CSV file.
        trip_ids: Set of trip_id values to include.

    returns:
        Mapping from trip_id to its stops sorted by stop_sequence.
    """
    trip_seq_rows = defaultdict(list)
    for row in read_dict_rows(stop_times_file):
        trip_id = row.get("trip_id", "").strip()
        if trip_id not in trip_ids:
            continue
        stop_id = row.get("stop_id", "").strip()
        if not stop_id:
            continue
        try:
            stop_sequence = int(row.get("stop_sequence", "").strip())
        except Exception:
            continue
        trip_seq_rows[trip_id].append((stop_sequence, stop_id))
    for trip_id in trip_seq_rows:
        trip_seq_rows[trip_id].sort(key=lambda item: item[0])
    return trip_seq_rows


def detect_canonical_violations(
    trip_seq_rows, trip_to_route_dir, rid_to_name, route_names_stop_ids
):
    """Return trips whose consecutive stops break the canonical route order.

    Only stop_sequence pairs with a gap of exactly one are checked. Pairs
    separated by a larger gap are considered intentionally non-adjacent and skipped.

    args:
        trip_seq_rows: Mapping from trip_id to sorted (seq, stop_id) rows.
        trip_to_route_dir: Mapping from trip_id to (route_id, direction_id).
        rid_to_name: Mapping from route_id to route short name.
        route_names_stop_ids: Mapping from route name to canonical stop order.

    returns:
        List of (route_name, route_id, trip_id, direction_id, bad_pairs).
    """
    flagged = []
    for trip_id, seq_rows in trip_seq_rows.items():
        route_id, direction_id = trip_to_route_dir.get(trip_id, (None, None))
        if not route_id:
            continue
        route_name = rid_to_name.get(route_id)
        expected_order = list(route_names_stop_ids.get(route_name, []))
        if not expected_order:
            continue
        if direction_id == "1":
            expected_order = list(reversed(expected_order))
        adjacency = build_expected_adjacency(expected_order)
        bad_pairs = []
        for index in range(len(seq_rows) - 1):
            seq_a, stop_a = seq_rows[index]
            seq_b, stop_b = seq_rows[index + 1]
            if seq_b != seq_a + 1:
                continue
            if adjacency.get(stop_a) != stop_b:
                bad_pairs.append((seq_a, stop_a, seq_b, stop_b))
        if bad_pairs:
            flagged.append((route_name, route_id, trip_id, direction_id, bad_pairs))
    return flagged


def print_canonical_violations(flagged, stop_names):
    """Print each flagged trip with its out-of-order stop pairs.

    args:
        flagged: Output of detect_canonical_violations.
        stop_names: Mapping from stop_id to stop_name.
    """
    for route_name, route_id, trip_id, direction_id, bad_pairs in flagged:
        print(
            f"- route={route_name!r} route_id={route_id} trip_id={trip_id}"
            f" direction={direction_id} bad_pairs={len(bad_pairs)}"
        )
        for seq_a, stop_a, seq_b, stop_b in bad_pairs:
            print(
                f"    bad order: [{seq_a}] {stop_a} ({stop_names.get(stop_a, '(no name)')}) -> "
                f"[{seq_b}] {stop_b} ({stop_names.get(stop_b, '(no name)')})"
            )

In [ ]:
def main():
    """Validate that subway trips follow the canonical stop sequence for their route."""
    rid_to_name = None
    trip_to_route_dir = None
    matched_rows = 0
    available_route_ids = None
    suffix = None
    trip_ids = None
    trip_seq_rows = None
    stop_names = None
    flagged = None
    total_bad_pairs = 0
    check_missing_files([TRIPS_FILE, STOP_TIMES_CLEANED_FILE, STOPS_FILE])

    print_file_disclaimer([
        (STOP_TIMES_CLEANED_FILE, 'stop_times'),
        (STOPS_FILE, 'stops'),
        (TRIPS_FILE, 'trips'),
    ])

    rid_to_name = {rid: name for name, rid in basics.subway_routes_names_ids.items()}
    trip_to_route_dir, matched_rows = build_trip_to_route_dir(TRIPS_FILE, rid_to_name)

    if not trip_to_route_dir:
        available_route_ids = sorted(
            {
                row.get("route_id", "").strip()
                for row in read_dict_rows(TRIPS_FILE)
                if row.get("route_id", "").strip()
            }
        )
        suffix = "" if len(available_route_ids) <= 20 else " ..."
        print("No subway trips found in 'trips' for the canonical mappings.")
        print(f"Trips rows matching subway route_ids: {matched_rows}")
        print(f"Available route_id values in 'trips': {available_route_ids[:20]}{suffix}")
        return

    trip_ids = set(trip_to_route_dir)
    trip_seq_rows = build_trip_seq_rows(STOP_TIMES_CLEANED_FILE, trip_ids)
    stop_names = load_stop_names(STOPS_FILE)
    flagged = detect_canonical_violations(
        trip_seq_rows, trip_to_route_dir, rid_to_name, basics.subway_route_names_stop_ids
    )

    os.makedirs(STOP_SEQUENCE_BASE, exist_ok=True)
    with open(WRONG_STOP_SEQUENCES_FILE, "w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(
            fh, fieldnames=["trip_id", "stop_a", "stop_b", "seq_a", "seq_b"]
        )
        writer.writeheader()
        for _, _, trip_id, _, bad_pairs in flagged:
            for seq_a, stop_a, seq_b, stop_b in bad_pairs:
                writer.writerow({
                    "trip_id": trip_id, "stop_a": stop_a, "stop_b": stop_b,
                    "seq_a": seq_a, "seq_b": seq_b,
                })
                total_bad_pairs += 1

    if not flagged:
        print("All checked subway trips follow an allowed contiguous stop sequence.")
        print(
            f"{Path(WRONG_STOP_SEQUENCES_FILE).name} (empty) written"
            f" to {Path(WRONG_STOP_SEQUENCES_FILE).relative_to(_project_root)}"
        )
        return

    print(f"FOUND {len(flagged)} trips with unexpected stop sequences:")
    print_canonical_violations(flagged, stop_names)
    print(
        f"\n{Path(WRONG_STOP_SEQUENCES_FILE).name} written"
        f" to {Path(WRONG_STOP_SEQUENCES_FILE).relative_to(_project_root)}"
        f" ({total_bad_pairs} rows)"
    )

main()

##### Verifying it after the sequence fix



After running `data_validation/processing/3_stop_sequence.py` we re-run the same check on `stop_times_sequence.txt`. The same trips will still appear as having non-canonical order (the stops themselves have not changed), but the bad pairs should now show non-consecutive stop_sequence numbers, confirming that the gap was correctly inserted.

In [ ]:
def main():
    """Verify canonical stop order in stop_times_sequence after the sequence fix."""
    rid_to_name = None
    trip_to_route_dir = None
    matched_rows = 0
    available_route_ids = None
    suffix = None
    trip_ids = None
    trip_seq_rows = None
    stop_names = None
    flagged = None
    check_missing_files([TRIPS_FILE, STOP_TIMES_FILE, STOPS_FILE])

    print_file_disclaimer([
        (STOP_TIMES_FILE, 'stop_times'),
        (STOPS_FILE, 'stops'),
        (TRIPS_FILE, 'trips'),
    ])

    rid_to_name = {rid: name for name, rid in basics.subway_routes_names_ids.items()}
    trip_to_route_dir, matched_rows = build_trip_to_route_dir(TRIPS_FILE, rid_to_name)

    if not trip_to_route_dir:
        available_route_ids = sorted(
            {
                row.get("route_id", "").strip()
                for row in read_dict_rows(TRIPS_FILE)
                if row.get("route_id", "").strip()
            }
        )
        suffix = "" if len(available_route_ids) <= 20 else " ..."
        print("No subway trips found in 'trips' for the canonical mappings.")
        print(f"Trips rows matching subway route_ids: {matched_rows}")
        print(f"Available route_id values in 'trips': {available_route_ids[:20]}{suffix}")
        return

    trip_ids = set(trip_to_route_dir)
    trip_seq_rows = build_trip_seq_rows(STOP_TIMES_FILE, trip_ids)
    stop_names = load_stop_names(STOPS_FILE)
    flagged = detect_canonical_violations(
        trip_seq_rows, trip_to_route_dir, rid_to_name, basics.subway_route_names_stop_ids
    )

    if not flagged:
        print("All checked subway trips follow an allowed contiguous stop sequence.")
        return

    print(f"FOUND {len(flagged)} trips with unexpected stop sequences:")
    print_canonical_violations(flagged, stop_names)

main()

#### Which stops have the same arrival and deparature_time?